# Used Module

In [1]:
# اتصال به گوگل درایو
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ماژول های مورد استفاده در برنامه
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
!pip install scikit-optimize
from skopt import BayesSearchCV

# Data Extraction

In [95]:
# خواندن داده ها از فایل csv
df = pd.read_csv('/content/drive/MyDrive/used_cars.csv')
# df.head()

# Data Review

In [ ]:
# بررسی اطلاعات داده ها و دیتافریم
df.shape
df.columns
df.dtypes
df.nunique()
df['fuel'].value_counts()
df['seller_type'].value_counts()
df['transmission'].value_counts()
df['owner'].value_counts()
df['seats'].value_counts()
df.info()
df.describe()


# Data Preprocessing Befor Data Visualization

## Converting Labeled Numerical Data Into Numerical Data

In [96]:
# تبدیل داده های اسمی به داده های عددی در ستون mileage
df['mileage_kmpl'] = pd.to_numeric(
    df['mileage'].astype('string').str.extract(r'(\d+\.?\d*)')[0],
    errors='coerce')
df.drop(columns=['mileage'],inplace=True)

# تبدیل داده های اسمی به داده های عددی در ستون engine
df['engine_cc'] = pd.to_numeric(
    df['engine'].astype('string').str.extract(r'(\d+)')[0],
    errors='coerce')
df.drop(columns=['engine'],inplace=True)

#تبدیل داده های اسمی به داده های عددی در ستون max_power
df['max_power_bhp'] = pd.to_numeric(
    df['max_power'].astype('string').str.extract(r'(\d+\.?\d*)')[0],
    errors='coerce')
df.drop(columns=['max_power'],inplace=True)



## Extracting Multiple Data Items From A Single Data Item

In [ ]:
# متد استخراج داده ها از ستون torque
def extract_torque(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan, np.nan])

    value = str(value).strip().lower()

    # استخراج مقدار گشتاور
    torque_match = pd.Series([value]).str.extract(r'(\d+\.?\d*)').iloc[0, 0]

    if pd.isna(torque_match):
        return pd.Series([np.nan, np.nan, np.nan])

    torque_value = float(torque_match)

    # تبدیل kgm به Nm
    if 'kgm' in value:
        torque_nm = torque_value * 9.80665
    else:
        torque_nm = torque_value

    # استخراج دور موتور
    rpm_match = pd.Series([value]).str.extract(
        r'@\s*([\d,]+)(?:\s*-\s*([\d,]+))?')

    if pd.isna(rpm_match.iloc[0, 0]):
        rpm_min = np.nan
        rpm_max = np.nan
    else:
        rpm_min = float(
            rpm_match.iloc[0, 0].replace(',', ''))

        if pd.isna(rpm_match.iloc[0, 1]):
            rpm_max = rpm_min
        else:
            rpm_max = float(
                rpm_match.iloc[0, 1].replace(',', ''))

    return pd.Series([torque_nm, rpm_min, rpm_max])

df[['torque_nm', 'torque_rpm_min', 'torque_rpm_max']] = (df['torque'].apply(extract_torque))
df.drop(columns=['torque'],inplace=True)
df.head()

## Cleaning Duplicate Data

In [ ]:
# تشخیص و پردازش داده های تکراری
df[df.duplicated()].count()
df.drop_duplicates(keep='first',inplace=True)
df[df.duplicated()].count()

## Outlier Cleaning

In [ ]:
# متد محاسبه فاصله بین چارکی داده های ستون ها و تشخیص داده های پرت بر اساس ان
def find_outliers(column):
  q1 = column.quantile(0.25)
  q3 = column.quantile(0.75)
  iqr = q3 - q1
  lower_bound = q1 - 1.5 * iqr
  upper_bound = q3 + 1.5 * iqr
  return column[(column < lower_bound) | (column > upper_bound)]

# جداسازی ستون های دارای داده عددی
df_number_data = df.select_dtypes(include=['number']).drop(columns=['year','seats'])
outliers_indexes = set()

# تشخیص داده های پرت موجود در ستون های عددی
for i , column in enumerate(df_number_data.columns):
  outliers = find_outliers(df[column])
  if (outliers.size > 0):
    print(f'Outliers - {column} : \n{outliers}')
    outliers_indexes.update(outliers.index)
  else :
    print(f'No Outliers - {column}')

# پاکسازی و حذف داده های پرت
df.drop(index=list(outliers_indexes),inplace=True)
print(f'\nTotal Rows Deleted : {len(outliers_indexes)}')


## Filling In Missing Data

In [ ]:
# تشخیص و پرکردن داده های خالی با مقدار میانگین ان ستون یا داده با بیشترین تکرار در ان ستون
df.isnull().sum()
df.fillna(value={
    'torque_nm' : df['torque_nm'].mean(),
    'torque_rpm_min' : df['torque_rpm_min'].mean(),
    'torque_rpm_max' : df['torque_rpm_max'].mean(),
    'seats' : df['seats'].mode()[0],
    'mileage_kmpl' : round(df['mileage_kmpl'].mean()),
    'engine_cc' : round(df['engine_cc'].mean()),
    'max_power_bhp' : round(df['max_power_bhp'].mean())
},inplace=True)
df.isnull().sum()

## Rounding Decimal Data To Two Decimal Places

In [ ]:
# محدود سازی رقم های اعشاری به 2 رقم اعشار در ستون های زیر
df[['torque_nm', 'torque_rpm_min', 'torque_rpm_max']] = df[
    ['torque_nm', 'torque_rpm_min', 'torque_rpm_max']].round(2)

## Converting Decimal Data To An Integer

In [44]:
# تبدیل داده های ستون صندلی ها به داده عدد صحیح
df['seats'] = df['seats'].astype(int)

# Data Visualization

## Chart Of Average Car Price by Year

In [ ]:
# ترسیم نمودار میانگین قیمت خودرو بر اساس سال
plt.figure(figsize=(12, 6))
sns.pointplot(
    data=df,
    x='year',
    y='selling_price',
    errorbar=None
)

plt.title('Average Car Price by Year')
plt.xlabel('Year')
plt.ylabel('Average Selling Price')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Chart Of Sales Volume Per Owner

In [ ]:
# ترسیم نمودار حجم فروش بر اساس تفکیک مالک
plt.figure(figsize=(12, 6))

plt_df = df.groupby(df['owner'])['selling_price'].count()
x = plt_df.index
y = plt_df.values

plt.bar(x,y)

plt.title('Sales Volume Per Owner')
plt.xlabel('Owner')
plt.ylabel('Sales Volume')

plt.tight_layout()
plt.show()

## Chart Of Distribution Of Car Selling Prices

In [ ]:
# ترسیم نمودار توزیع قیمت های فروش خودرو
plt.figure(figsize=(12, 6))

sns.histplot(
    data=df,
    x='selling_price',
    bins=40,
    kde=True)

plt.title('Distribution Of Car Selling Prices')
plt.xlabel('Selling Price')
plt.ylabel('Frequency')

plt.ticklabel_format(style='plain', axis='x')

plt.tight_layout()
plt.show()

## Chart Of Year vs Selling Price

In [ ]:
# ترسیم نمودار سال در برابر قیمت فروش
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=df,
    x='year',
    y='selling_price',
    alpha=0.5)

plt.title('Year vs Selling Price')
plt.xlabel('Year')
plt.ylabel('Selling Price')

plt.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

## Chart Of Kilometers Driven vs Selling Price

In [ ]:
# ترسیم نمودار مسافت طی‌شده (به کیلومتر) در برابر قیمت فروش
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=df,
    x='km_driven',
    y='selling_price',
    alpha=0.4)

plt.title('Kilometers Driven vs Selling Price')
plt.xlabel('Kilometers Driven')
plt.ylabel('Selling Price')

plt.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

## Chart Of Selling Price by Transmission

In [ ]:
# ترسیم نمودار قیمت فروش بر اساس نوع جعبه‌دنده
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x='transmission',
    y='selling_price')

plt.title('Selling Price by Transmission')
plt.xlabel('Transmission')
plt.ylabel('Selling Price')

plt.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

## Chart Of Selling Price by Seller Type

In [ ]:
# ترسیم نمودار قیمت فروش بر اساس نوع فروشنده
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x='seller_type',
    y='selling_price')

plt.title('Selling Price by Seller Type')
plt.xlabel('Seller Type')
plt.ylabel('Selling Price')

plt.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

# Continuation Of Data Preprocessing

## Converting Km_Driven Values

In [ ]:
# محاسبه لگاریتم و میانگین مربعات خطا برای داده های ستون km_driven
df['km_driven_log'] = np.log(df['km_driven'])
df['km_driven_sqrt'] = np.sqrt(df['km_driven'])

from scipy.stats import skew

# محاسبه و نمایش میزان چولگی برای هر 3 مدل داده ( لگاریتمی ، میانگین مربعات خطا و اصلی ) در ستون km_driven
print(f'Skewness of km_driven : {skew(df["km_driven"])}')
print(f'Skewness of km_driven_log : {skew(df["km_driven_log"])}')
print(f'Skewness of km_driven_sqrt : {skew(df["km_driven_sqrt"])}')

In [ ]:
# ترسیم نمودار از هر 3 مدل داده ( لگاریتمی ، میانگین مربعات خطا و اصلی ) در ستون km_driven
fig , axes = plt.subplots(1,3,figsize=(15,5))
sns.histplot(data=df,x='km_driven',ax=axes[0])
sns.histplot(data=df,x='km_driven_log',ax=axes[1])
sns.histplot(data=df,x='km_driven_sqrt',ax=axes[2])
axes[0].set_title('km_driven')
axes[1].set_title('km_driven_log')
axes[2].set_title('km_driven_sqrt')
plt.tight_layout()
plt.show()

In [ ]:
# تبدیل داده های ستون اصلی به بینه ترین حالت داده های ان ستون
df['km_driven'] = df['km_driven_sqrt']
df.drop(columns=['km_driven_log','km_driven_sqrt'],inplace=True)
df

## Converting Selling_Price Values

In [ ]:
# محاسبه لگاریتم و میانگین مربعات خطا برای داده های ستون selling_price
df['selling_price_log'] = np.log(df['selling_price'])
df['selling_price_sqrt'] = np.sqrt(df['selling_price'])

from scipy.stats import skew

# محاسبه و نمایش میزان چولگی برای هر 3 مدل داده ( لگاریتمی ، میانگین مربعات خطا و اصلی ) در ستون selling_price
print(f'Skewness of selling_price : {skew(df["selling_price"])}')
print(f'Skewness of selling_price_log : {skew(df["selling_price_log"])}')
print(f'Skewness of selling_price_sqrt : {skew(df["selling_price_sqrt"])}')

In [ ]:
# ترسیم نمودار از هر 3 مدل داده ( لگاریتمی ، میانگین مربعات خطا و اصلی ) در ستون selling_price
fig , axes = plt.subplots(1,3,figsize=(15,5))
sns.histplot(data=df,x='selling_price',ax=axes[0])
sns.histplot(data=df,x='selling_price_log',ax=axes[1])
sns.histplot(data=df,x='selling_price_sqrt',ax=axes[2])
axes[0].set_title('selling_price')
axes[1].set_title('selling_price_log')
axes[2].set_title('selling_price_sqrt')
plt.tight_layout()
plt.show()

In [ ]:
# تبدیل داده های ستون اصلی به بینه ترین حالت داده های ان ستون
df['selling_price'] = df['selling_price_sqrt']
df.drop(columns=['selling_price_log','selling_price_sqrt'],inplace=True)
df.head()

## Converting Categorical Data To Numerical Data Using Label Encoder

In [ ]:
# دسته بندی و کدگذاری داده های اسمی و رشته ای به روش Label Encoder
le = LabelEncoder()
df['name'] = le.fit_transform(df['name'])
df['fuel'] = le.fit_transform(df['fuel'])
df['seller_type'] = le.fit_transform(df['seller_type'])
df['transmission'] = le.fit_transform(df['transmission'])
df['owner'] = le.fit_transform(df['owner'])
df.describe()

# Splitting Data Into Training And Testing Sets And Models Training And Evaluation

## For Univariate Regression

In [ ]:
# ایجاد داده های متغیر و هدف برای اموزش مدل تک متغییره
x = df[['year']]
y = df['selling_price']
x.shape , y.shape

In [ ]:
# تقسیم داده های موجود به داده های مخصوص آموزش و داده های مخصوص ارزیابی
X_train , X_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42)
X_train.shape , X_test.shape , y_train.shape , y_test.shape

In [ ]:
# مقیاس بندی داده های متغییر و محاسبه میانگین و انحراف معیار داده های مخصوص آموزش
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(f'Mean :',X_train.mean(axis=0))
print(f'Standard Deviation :',X_train.std(axis=0))

In [ ]:
# ساخت مدل رگرسیون نک متغیره و آموزش ان با داده های مخصوص آموزش
lr1_model = LinearRegression()
lr1_model.fit(X_train,y_train)

# ارزیابی عملکرد مدل با داده های مخصوص ارزیابی و معیار های میانگین مربعات خطا و نمره R2
y_pred = lr1_model.predict(X_test)
lr1_mse = mean_squared_error(y_test,y_pred)
lr1_r2 = r2_score(y_test,y_pred)
print(f'Mean Squared Error (Univariate Liner Regression) : {lr1_mse}')
print(f'R2 Score (Univariate Liner Regression) : {lr1_r2}')


## For Multivariate Regression

In [ ]:
# جداسازی ستون داده هدف از متغیر های دیگر
x = df.drop(columns=['selling_price'])
y = df['selling_price']
x.shape , y.shape

In [ ]:
# تقسیم داده های موجود به داده های مخصوص آموزش و داده های مخصوص ارزیابی
X_train , X_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42)
X_train.shape , X_test.shape , y_train.shape , y_test.shape

In [ ]:
# مقیاس بندی داده های متغییر و محاسبه میانگین و انحراف معیار داده های مخصوص آموزش
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(f'Mean :',X_train.mean(axis=0))
print(f'Standard Deviation :',X_train.std(axis=0))

In [ ]:
# ساخت مدل رگرسیون چند متغیره و آموزش ان با داده های مخصوص آموزش
lrn_model = LinearRegression()
lrn_model.fit(X_train,y_train)

# ارزیابی عملکرد مدل با داده های مخصوص ارزیابی و معیار های میانگین مربعات خطا و نمره R2
y_pred = lrn_model.predict(X_test)
lrn_mse = mean_squared_error(y_test,y_pred)
lrn_r2 = r2_score(y_test,y_pred)
print(f'Mean Squared Error (Multivariate Liner Regression) : {lrn_mse}')
print(f'R2 Score (Multivariate Liner Regression) : {lrn_r2}')

## For Polynomial Regression

In [ ]:
# جداسازی ستون داده هدف از متغیر های دیگر
x = df.drop(columns=['selling_price'])
y = df['selling_price']
x.shape , y.shape

In [ ]:
# تقسیم داده های موجود به داده های مخصوص آموزش و داده های مخصوص ارزیابی
X_train , X_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42)
X_train.shape , X_test.shape , y_train.shape , y_test.shape

In [ ]:
# مقیاس بندی داده های متغییر و محاسبه میانگین و انحراف معیار داده های مخصوص آموزش
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(f'Mean :',X_train.mean(axis=0))
print(f'Standard Deviation :',X_train.std(axis=0))

In [ ]:
# ساخت مدل رگرسیون چند جمله ای و ایجاد و ترکیب متغیر های داده های مخصوص آموزش و ارزیابی
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# ساخت مدل رگرسیون چند متغیره و آموزش ان با داده های مخصوص آموزش و ارزیابی ایجاد شده توسط مدل رگرسون چندجمله ای
poly_model = LinearRegression()
poly_model.fit(X_train_poly,y_train)

# ارزیابی عملکرد مدل با داده های مخصوص ارزیابی و معیار های میانگین مربعات خطا و نمره R2
y_pred = poly_model.predict(X_test_poly)
poly_mse = mean_squared_error(y_test, y_pred)
poly_r2 = r2_score(y_test, y_pred)
print(f'Mean Squared Error (Polynomial Regression) : {poly_mse}')
print(f'R2 Score (Polynomial Regression) : {poly_r2}')

## Finding The Best Hyperparameters With Random Search For Ridge Regression

In [ ]:
# ساخت مدل رگرسون Ridge
ridge_model_test = Ridge()

# ایجاد پارامتر های مورد نظر برای یافتن بهترین انها
params = {'alpha' : [1e-4,1e-3,1e-2,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]}

# ساخت و ارزیابی مدل با تمامی پارامتر های مورد نظر به روش Randomized Search
random_search = RandomizedSearchCV(
                                    estimator=ridge_model_test,
                                    param_distributions=params,
                                    scoring='r2',
                                    cv=5,
                                    n_iter=25,
                                    random_state=42)

# آموزش مدل ساخته شده با داده های مخصوص آموزش
random_search.fit(X_train,y_train)

# به دست اوردن بهترین پارامتر ها و بهترین  نمره R2
print(f'Randomaized Search CV :\n===================================')
print(f'Best Parameters : {random_search.best_params_}')
print(f'Best R2 Score : {random_search.best_score_}')
print(40*'=')

# ساخت مدل رگرسیون Ridge با بهترین پارمتر به دست امده و آموزش ان با داده های مخصوص آموزش
ridge_model = Ridge(alpha=random_search.best_params_['alpha'])
ridge_model.fit(X_train,y_train)

# ارزیابی عملکرد مدل با داده های مخصوص ارزیابی و معیار های میانگین مربعات خطا و نمره R2
y_pred = ridge_model.predict(X_test)
ridge_mse = mean_squared_error(y_test,y_pred)
ridge_r2 = r2_score(y_test,y_pred)
print(f'Mean Squared Error (Ridge Regression) : {ridge_mse}')
print(f'R2 Score (Ridge Regression) : {ridge_r2}')

## Finding The Best Hyperparameters With Bayesian Optimization For Lasso Regression

In [ ]:
# ساخت مدل رگرسون Lasso
lasso_model_test = Lasso()

# ایجاد پارامتر های مورد نظر برای یافتن بهترین انها
params = {'alpha' : [1e-4,1e-3,1e-2,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]}

# ساخت و ارزیابی مدل با تمامی پارامتر های مورد نظر به روش Bayes Search
opt = BayesSearchCV(
                    estimator=lasso_model_test,
                    search_spaces=params,
                    scoring='r2',
                    cv=5,
                    n_iter=25,
                    random_state=4)

# آموزش مدل ساخته شده با داده های مخصوص آموزش
opt.fit(X_train,y_train)

# به دست اوردن بهترین پارامتر ها و بهترین  نمره R2
print(f'Bayesian Optimization :\n===================================')
print(f'Best Parameters : {opt.best_params_}')
print(f'Best R2 Score : {opt.best_score_}')
print(40*'=')

# ساخت مدل رگرسیون Lasso با بهترین پارمتر به دست امده و آموزش ان با داده های مخصوص آموزش
lasso_model = Lasso(alpha=opt.best_params_['alpha'])
lasso_model.fit(X_train,y_train)

# ارزیابی عملکرد مدل با داده های مخصوص ارزیابی و معیار های میانگین مربعات خطا و نمره R2
y_pred = lasso_model.predict(X_test)
lasso_mse = mean_squared_error(y_test,y_pred)
lasso_r2 = r2_score(y_test,y_pred)
print(f'Mean Squared Error (Lasso Regression) : {lasso_mse}')
print(f'R2 Score (Lasso Regression) : {lasso_r2}')

## Finding The Best Hyperparameters With Cross Validation For Elastic Net Regression

In [ ]:
# ایجاد پارامتر های مورد نظر برای یافتن بهترین انها
alphas = [1e-4,1e-3,1e-2,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
l1_ratios = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

# ایجاد حالت های مختلف از ترکیب مقادیر پارامتر های مورد نظر برای یافتن بهترین انها
params_dict = [{'alpha': alpha, 'l1_ratio': l1_ratio} for alpha in alphas for l1_ratio in l1_ratios]

# ساخت مدل و ارزیابی مدل با تمامی حالت های موجود از ترکیب پارامتر های مورد نظر
models_test = [ElasticNet(alpha=param['alpha'],l1_ratio=param['l1_ratio']) for param in params_dict]

# آموزش و ارزیابی مدل با تمامی پارامتر های مورد نظر به روش Cross Validation
cv_scores = [cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean() for model in models_test]

# به دست اوردن بهترین پارامتر ها و بهترین  نمره R2
best_index = np.argmax(cv_scores)
best_params = params_dict[best_index]
print('Cross Validation :\n===================================')
print(f'Best Parameters : {best_params}')
print(f'Best R2 Score : {cv_scores[best_index]}')
print(40*'=')

# ساخت مدل رگرسیون ElasticNet با بهترین پارمتر های به دست امده و آموزش ان با داده های مخصوص آموزش
elastic_model = ElasticNet(alpha=best_params['alpha'],l1_ratio=best_params['l1_ratio'])
elastic_model.fit(X_train,y_train)

# ارزیابی عملکرد مدل با داده های مخصوص ارزیابی و معیار های میانگین مربعات خطا و نمره R2
y_pred = elastic_model.predict(X_test)
elastic_mse = mean_squared_error(y_test,y_pred)
elastic_r2 = r2_score(y_test,y_pred)
print(f'Mean Squared Error (Elastic Net Regression) : {elastic_mse}')
print(f'R2 Score (Elastic Net Regression) : {elastic_r2}')

# Chart Comparing The Obtained Results

In [ ]:
# ایجاد دیتافریم از نتایج و نمرات به دست امده از همه مدل های رگرسیون
results = pd.DataFrame({
    'Model': ['Univariate Linear Regression',
              'Multivariate Linear Regression',
              'Polynomial Regression',
              'Ridge Regression',
              'Lasso Regression',
              'Elastic Net Regression'],
    'MSE': [lr1_mse,
                           lrn_mse,
                           poly_mse,
                           ridge_mse,
                           lasso_mse,
                           elastic_mse],
    'R2_Score': [lr1_r2,
                 lrn_r2,
                 poly_r2,
                 ridge_r2,
                 lasso_r2,elastic_r2]})
results

In [ ]:
# ترسیم نمودار و مقایسه نتایج هر مدل برای نمره R2
plt.figure(figsize=(12, 6))
plt.bar(results['Model'], results['R2_Score'])

plt.title('R2 Score Comparison')
plt.xlabel('Models')
plt.ylabel('R2 Score')

plt.xticks(rotation=30, ha='right')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ترسیم نمودار و مقایسه نتایج هر مدل برای میانگین مربعات خطا
plt.figure(figsize=(12, 6))

plt.bar(results['Model'], results['MSE'])

plt.title('MSE Comparison')
plt.xlabel('Models')
plt.ylabel('MSE')

plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()